# Load, plot, and animate spin-resolved 2D B-sweep simulations

This notebook is the post-processing companion for
`sweeppone2_spin_exchange_verified.ipynb`.

It loads the folders generated by the spin/exchange B sweep. Each saved run is expected to contain:

- `metadata.json`, with `params["B"]` and sweep metadata.
- `data.npz`, with arrays such as `U_scalar`, `U_up`, `U_down`, `n`, `n_up`, `n_down`, `nu`, `nu_up`, `nu_down`, `spin_polarization`, `Vx_up`, and `Vx_down`.

The notebook can make videos of scalar electrostatic potential, spin-dependent potentials, spin densities, filling factors, exchange potentials, and spin polarization.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import HTML, display

plt.rcParams.update({"font.size": 12})

## User options

Set `SWEEP_DIR` to the directory generated by `sweeppone2_spin_exchange_verified.ipynb`.
The default below matches the latest generated sweep notebook.

The most useful movie keys are:

- `U_scalar`: scalar electrostatic effective potential, compatible with the old spinless `U` diagnostic.
- `U_up`, `U_down`: spin-dependent effective potentials including exchange.
- `n`, `n_up`, `n_down`: total and spin-resolved densities.
- `nu`, `nu_up`, `nu_down`: total and spin-resolved local filling factors.
- `spin_polarization`: \((n_\uparrow-n_\downarrow)/(n_\uparrow+n_\downarrow)\).
- `Vx_up`, `Vx_down`: local exchange potentials.

In [ ]:
# Directory produced by the verified spin/exchange sweep notebook.
#SWEEP_DIR = Path("sweep_fold_spin_exchange")
SWEEP_DIR = Path("results_2d_half_top_gate_spin_exchange")

# Which fields to load into memory. Keep this list modest for very large sweeps.
ARRAY_KEYS = [
    "U_scalar", "U_up", "U_down",
    "n", "n_up", "n_down",
    "nu", "nu_up", "nu_down",
    "spin_polarization",
    "Vx_up", "Vx_down",
]

# Backward-compatible fallbacks. If a key is missing, the loader tries these alternatives.
FALLBACK_KEYS = {
    "U_scalar": ["U", "U_ext"],
    "n": ["density"],
    "nu": ["filling"],
}

GROUND_KEYS = {"U_scalar", "U_up", "U_down", "U", "U_ext"}
GROUND_EACH_FRAME = True       # subtract min of each potential frame for potential-like arrays
GROUND_ONLY_INSIDE_MASK = True  # use mask when available for the minimum
DTYPE = np.float32

# Cache and video outputs
CACHE_FILE = Path("spin_exchange_sweep_cache.npz")
VIDEO_DIR = Path("spin_exchange_videos")
VIDEO_DIR.mkdir(exist_ok=True)

FPS = 10
DPI = 150
CMAP = "viridis"
SPIN_CMAP = "coolwarm"

# Default common color limits. Use None for percentile-based auto limits.
DEFAULT_PERCENTILES = (1, 99)
PLOT_LIMITS = {
    "U_scalar": (0, 10),
    "U_up": None,
    "U_down": None,
    "n": None,
    "n_up": None,
    "n_down": None,
    "nu": None,
    "nu_up": None,
    "nu_down": None,
    "spin_polarization": (-1, 1),
    "Vx_up": None,
    "Vx_down": None,
}

## Discover saved simulations

The loader searches recursively below `SWEEP_DIR` for folders containing both `metadata.json` and `data.npz`, then sorts by magnetic field `B`.

In [ ]:
def find_simulation_dirs(sweep_dir: Path):
    sweep_dir = Path(sweep_dir)
    if not sweep_dir.exists():
        raise FileNotFoundError(
            f"Cannot find {sweep_dir!s}. Run this notebook from the directory containing the sweep folder, "
            "or set SWEEP_DIR to the correct path."
        )
    run_dirs = []
    for data_file in sweep_dir.rglob("data.npz"):
        run_dir = data_file.parent
        if (run_dir / "metadata.json").exists():
            run_dirs.append(run_dir)
    if not run_dirs:
        raise FileNotFoundError(f"No simulation folders with data.npz and metadata.json found below {sweep_dir!s}.")
    return sorted(run_dirs)


def read_metadata(run_dir: Path):
    with open(run_dir / "metadata.json", "r") as f:
        return json.load(f)


def read_B_from_metadata(run_dir: Path):
    metadata = read_metadata(run_dir)
    try:
        return float(metadata["params"]["B"]), metadata
    except KeyError as exc:
        # Compatibility with possible older sweep metadata.
        if "extra_metadata" in metadata and "sweep_B" in metadata["extra_metadata"]:
            return float(metadata["extra_metadata"]["sweep_B"]), metadata
        raise KeyError(f"Could not find params['B'] in {run_dir / 'metadata.json'}") from exc


run_dirs = find_simulation_dirs(SWEEP_DIR)
print(f"Found {len(run_dirs)} saved simulations.")
print("First folder:", run_dirs[0])
print("Last folder: ", run_dirs[-1])

# Show available keys from the first run.
with np.load(run_dirs[0] / "data.npz", allow_pickle=False) as data0:
    print("Available data keys in first run:")
    print(data0.files)

## Load selected spin-resolved sequences

Only the requested `ARRAY_KEYS` are stacked. Coordinates and masks are read once and checked for shape compatibility.

In [ ]:
def _resolve_key(data, requested_key, fallback_keys=None):
    if requested_key in data.files:
        return requested_key
    for candidate in (fallback_keys or {}).get(requested_key, []):
        if candidate in data.files:
            return candidate
    return None


def _ground_frame(arr, mask=None):
    if mask is not None and np.any(mask):
        return arr - np.nanmin(arr[mask])
    return arr - np.nanmin(arr)


def load_spin_exchange_sequence(
    run_dirs,
    array_keys,
    fallback_keys=None,
    ground_each_frame=True,
    ground_keys=None,
    ground_only_inside_mask=True,
    dtype=np.float32,
):
    records = []
    x_ref = y_ref = mask_ref = None
    shape_ref = None
    metadata_list = []

    for run_dir in run_dirs:
        B, metadata = read_B_from_metadata(run_dir)
        loaded = {}
        with np.load(run_dir / "data.npz", allow_pickle=False) as data:
            # read coordinates and mask
            if x_ref is None:
                # infer shape from the first requested/available 2D array
                first_key = None
                for key in array_keys:
                    first_key = _resolve_key(data, key, fallback_keys)
                    if first_key is not None:
                        break
                if first_key is None:
                    raise KeyError(f"None of ARRAY_KEYS={array_keys} found in {run_dir / 'data.npz'}")
                first_arr = np.asarray(data[first_key])
                shape_ref = first_arr.shape
                x_ref = np.asarray(data["x"], dtype=dtype) if "x" in data.files else np.arange(shape_ref[1], dtype=dtype)
                y_ref = np.asarray(data["y"], dtype=dtype) if "y" in data.files else np.arange(shape_ref[0], dtype=dtype)
                mask_ref = np.asarray(data["mask"], dtype=bool) if "mask" in data.files else np.ones(shape_ref, dtype=bool)

            for key in array_keys:
                actual_key = _resolve_key(data, key, fallback_keys)
                if actual_key is None:
                    print(f"Warning: key {key!r} not found in {run_dir.name}; filling with NaN.")
                    arr = np.full(shape_ref, np.nan, dtype=dtype)
                else:
                    arr = np.asarray(data[actual_key], dtype=dtype)
                    if arr.shape != shape_ref:
                        raise ValueError(f"Grid shape mismatch for key {actual_key} in {run_dir}: {arr.shape} != {shape_ref}")
                    if ground_each_frame and key in (ground_keys or set()):
                        mask_for_min = mask_ref if ground_only_inside_mask else None
                        arr = _ground_frame(arr, mask_for_min)
                loaded[key] = arr.astype(dtype, copy=False)

        records.append((B, run_dir, loaded, metadata))

    records.sort(key=lambda item: item[0])
    B_values = np.array([r[0] for r in records], dtype=dtype)
    sorted_dirs = [r[1] for r in records]
    metadata_list = [r[3] for r in records]
    stacks = {
        key: np.stack([r[2][key] for r in records], axis=0).astype(dtype, copy=False)
        for key in array_keys
    }
    return B_values, stacks, x_ref, y_ref, mask_ref, sorted_dirs, metadata_list


B_values, stacks, x, y, mask, sorted_run_dirs, metadata_list = load_spin_exchange_sequence(
    run_dirs,
    ARRAY_KEYS,
    fallback_keys=FALLBACK_KEYS,
    ground_each_frame=GROUND_EACH_FRAME,
    ground_keys=GROUND_KEYS,
    ground_only_inside_mask=GROUND_ONLY_INSIDE_MASK,
    dtype=DTYPE,
)

print("B_values.shape:", B_values.shape)
print("Loaded stacks:")
for key, arr in stacks.items():
    print(f"  {key:18s}: {arr.shape}, {arr.nbytes/1024**2:.2f} MB")
print("x.shape, y.shape:", x.shape, y.shape)
print(f"B range: {B_values.min():.6g} -> {B_values.max():.6g} T")

## Save a lightweight cache

The cache stores only the loaded sequences and metadata useful for plotting. It can be regenerated from the sweep folder at any time.

In [ ]:
np.savez_compressed(
    CACHE_FILE,
    B_values=B_values,
    x=x,
    y=y,
    mask=mask,
    array_keys=np.array(ARRAY_KEYS),
    **{f"stack__{key}": value for key, value in stacks.items()},
)
print(f"Saved lightweight cache to {CACHE_FILE.resolve()}")

## Plot utilities

In [ ]:
def label_for_key(key):
    labels = {
        "U_scalar": r"$U_{scalar}-\min(U_{scalar})$ [meV]" if GROUND_EACH_FRAME else r"$U_{scalar}$ [meV]",
        "U_up": r"$U_\uparrow-\min(U_\uparrow)$ [meV]" if GROUND_EACH_FRAME else r"$U_\uparrow$ [meV]",
        "U_down": r"$U_\downarrow-\min(U_\downarrow)$ [meV]" if GROUND_EACH_FRAME else r"$U_\downarrow$ [meV]",
        "n": r"$n$ [nm$^{-2}$]",
        "n_up": r"$n_\uparrow$ [nm$^{-2}$]",
        "n_down": r"$n_\downarrow$ [nm$^{-2}$]",
        "nu": r"$\nu$",
        "nu_up": r"$\nu_\uparrow$",
        "nu_down": r"$\nu_\downarrow$",
        "spin_polarization": r"$(n_\uparrow-n_\downarrow)/(n_\uparrow+n_\downarrow)$",
        "Vx_up": r"$V^x_\uparrow$ [meV]",
        "Vx_down": r"$V^x_\downarrow$ [meV]",
    }
    return labels.get(key, key)


def cmap_for_key(key):
    if key in {"spin_polarization", "Vx_up", "Vx_down"}:
        return SPIN_CMAP
    return CMAP


def color_limits_for_key(key, stack, percentiles=DEFAULT_PERCENTILES):
    if key in PLOT_LIMITS and PLOT_LIMITS[key] is not None:
        return PLOT_LIMITS[key]
    finite = stack[np.isfinite(stack)]
    if finite.size == 0:
        return None, None
    return tuple(float(v) for v in np.nanpercentile(finite, percentiles))


def plot_frame(key="U_scalar", index=0, vmin=None, vmax=None, ax=None):
    stack = stacks[key]
    if vmin is None and vmax is None:
        vmin, vmax = color_limits_for_key(key, stack)
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7.0, 3.8))
    else:
        fig = ax.figure
    extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]
    im = ax.imshow(
        stack[index],
        origin="lower",
        extent=extent,
        aspect="auto",
        cmap=cmap_for_key(key),
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"{key}, B = {B_values[index]:.4g} T")
    ax.set_xlabel("x [nm]")
    ax.set_ylabel("y [nm]")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(label_for_key(key))
    fig.tight_layout()
    return fig, ax, im

## Inspect representative frames

In [ ]:
KEY_TO_PREVIEW = "U_scalar"
preview_indices = np.linspace(0, len(B_values)-1, min(4, len(B_values)), dtype=int)
vmin, vmax = color_limits_for_key(KEY_TO_PREVIEW, stacks[KEY_TO_PREVIEW])
for idx in preview_indices:
    plot_frame(KEY_TO_PREVIEW, idx, vmin=vmin, vmax=vmax)
plt.show()

## Compare spin channels for a selected frame

In [ ]:
FRAME_INDEX = 0
COMPARE_KEYS = ["n_up", "n_down", "spin_polarization"]

fig, axes = plt.subplots(1, len(COMPARE_KEYS), figsize=(5.2*len(COMPARE_KEYS), 3.8), constrained_layout=True)
if len(COMPARE_KEYS) == 1:
    axes = [axes]
extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]
for ax, key in zip(axes, COMPARE_KEYS):
    vmin, vmax = color_limits_for_key(key, stacks[key])
    im = ax.imshow(stacks[key][FRAME_INDEX], origin="lower", extent=extent, aspect="auto", cmap=cmap_for_key(key), vmin=vmin, vmax=vmax)
    ax.set_title(f"{key}, B={B_values[FRAME_INDEX]:.4g} T")
    ax.set_xlabel("x [nm]")
    ax.set_ylabel("y [nm]")
    fig.colorbar(im, ax=ax, label=label_for_key(key))
plt.show()

## Build and preview one animation

Change `VIDEO_KEY` to animate another quantity.

In [ ]:
def make_animation(key="U_scalar", vmin=None, vmax=None, fps=FPS):
    stack = stacks[key]
    if vmin is None and vmax is None:
        vmin, vmax = color_limits_for_key(key, stack)

    fig, ax = plt.subplots(figsize=(7.0, 3.8))
    extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]
    im = ax.imshow(
        stack[0],
        origin="lower",
        extent=extent,
        aspect="auto",
        cmap=cmap_for_key(key),
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel("x [nm]")
    ax.set_ylabel("y [nm]")
    title = ax.set_title("")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(label_for_key(key))
    fig.tight_layout()

    def update(frame):
        im.set_data(stack[frame])
        title.set_text(f"{key}, B = {B_values[frame]:.4g} T   frame {frame+1}/{len(B_values)}")
        return im, title

    anim = FuncAnimation(fig, update, frames=len(B_values), interval=1000/fps, blit=False)
    plt.close(fig)
    return anim


VIDEO_KEY = "U_scalar"
anim = make_animation(VIDEO_KEY)
display(HTML(anim.to_jshtml()))

## Export videos

This cell exports one MP4/GIF for each key in `VIDEO_KEYS_TO_EXPORT`. MP4 is attempted first; if `ffmpeg` is unavailable, the notebook falls back to GIF.

In [ ]:
VIDEO_KEYS_TO_EXPORT = ["U_scalar", "n", "n_up", "n_down", "nu", "spin_polarization", "Vx_up", "Vx_down"]


def export_animation(key, video_dir=VIDEO_DIR, fps=FPS, dpi=DPI):
    anim = make_animation(key, fps=fps)
    mp4_path = Path(video_dir) / f"{key}_sweep.mp4"
    gif_path = Path(video_dir) / f"{key}_sweep.gif"
    try:
        writer = FFMpegWriter(fps=fps, bitrate=2400)
        anim.save(mp4_path, writer=writer, dpi=dpi)
        print(f"Saved MP4: {mp4_path.resolve()}")
        return mp4_path
    except Exception as exc:
        print(f"MP4 export failed for {key}; falling back to GIF. Error: {exc!r}")
        writer = PillowWriter(fps=fps)
        anim.save(gif_path, writer=writer, dpi=dpi)
        print(f"Saved GIF: {gif_path.resolve()}")
        return gif_path


exported_files = []
for key in VIDEO_KEYS_TO_EXPORT:
    if key in stacks:
        exported_files.append(export_animation(key))
print("Exported files:")
for path in exported_files:
    print(" -", path)

## 1D diagnostics along the center line

This reproduces the old diagnostic style but also includes spin-resolved quantities.

In [ ]:
y0_index = np.argmin(np.abs(y - 0.0))
frame_index = 0

fig, ax = plt.subplots(figsize=(7.0, 4.0))
for key in ["U_scalar", "U_up", "U_down"]:
    if key in stacks:
        ax.plot(x, stacks[key][frame_index, y0_index], label=key)
ax.set_title(f"Center-line potentials, B={B_values[frame_index]:.4g} T")
ax.set_xlabel("x [nm]")
ax.set_ylabel("potential [meV], grounded if enabled")
ax.grid(True)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 4.0))
for key in ["nu", "nu_up", "nu_down"]:
    if key in stacks:
        ax.plot(x, stacks[key][frame_index, y0_index], label=key)
ax.set_title(f"Center-line filling factors, B={B_values[frame_index]:.4g} T")
ax.set_xlabel("x [nm]")
ax.set_ylabel("filling")
ax.grid(True)
ax.legend()
plt.show()

## Potential-step diagnostic versus B

By default this uses `U_scalar`, matching the latest sweep notebook diagnostic. You can switch to `U_up` or `U_down` if you want the spin-dependent effective potential step.

In [ ]:
DIAGNOSTIC_KEY = "U_scalar"
x_right_probe_nm = 200.0
x_left_probe_nm = 0.0
y_probe_nm = 0.0

nx_right = np.argmin(np.abs(x - x_right_probe_nm))
nx_left = np.argmin(np.abs(x - x_left_probe_nm))
ny = np.argmin(np.abs(y - y_probe_nm))

dU = stacks[DIAGNOSTIC_KEY][:, ny, nx_right] - stacks[DIAGNOSTIC_KEY][:, ny, nx_left]

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(B_values, dU, marker="o", ms=3)
ax.set_xlabel("B [T]")
ax.set_ylabel(f"Δ{DIAGNOSTIC_KEY} [meV]")
ax.set_title(f"Potential step: x={x_right_probe_nm:g} nm minus x={x_left_probe_nm:g} nm")
ax.grid(True)
plt.show()

# Optional normalization by hbar omega_c ≈ 1.75 meV/T for GaAs-like parameters, as in the old notebook.
fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(B_values, dU/(1.75*B_values), marker="o", ms=3)
ax.set_xlabel("B [T]")
ax.set_ylabel(r"$\Delta U/(\hbar\omega_c)$")
ax.set_title("Dimensionless potential-step diagnostic")
ax.grid(True)
plt.show()

## Notes

- The latest sweep notebook saves `U_scalar`, `U_up`, `U_down`, `n_up`, `n_down`, `nu_up`, `nu_down`, `spin_polarization`, `Vx_up`, and `Vx_down`.
- `U_scalar` is the correct default for comparison with the old spinless electrostatic potential.
- `U_up` and `U_down` include the local exchange potential and are therefore spin-dependent.
- For very large sweeps, reduce `ARRAY_KEYS` before loading to avoid storing unnecessary 3D stacks in memory.